## Terminology Update

Some historical output logs in this notebook were captured before the current naming migration.
Current runtime terminology in the codebase is:
- `single_csv`
- `multi_csv`
- `ExecutionContext`


In [12]:
# Setup: add project root
import sys
import os
sys.path.insert(0, '..')

from src.orchestrator import Orchestrator
from src.standards import METADATA_STANDARDS
from src.context.context_factory import create_context
BASE = os.path.abspath(os.path.join('..'))

import logging

# Setup logging for Jupyter notebook
# Force reconfigure by removing existing handlers
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Remove existing handlers to avoid duplicates
for handler in logger.handlers[:]:
    logger.removeHandler(handler)

# Add a fresh StreamHandler that outputs to stdout (visible in notebook)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(name)s - %(message)s'))
logger.addHandler(handler)

STANDARD_NAME = "croissant_standard_subset"
# STANDARD_NAME = "spatial_ecological"

print("Imports OK")

Imports OK


In [13]:
from src.orchestrator import run_metadata_extraction
# multi_source = {
#     'biota': os.path.join(BASE, 'data/biota_dataset/biota.csv'),
#     'samples': os.path.join(BASE, 'data/biota_dataset/samples.csv'),
#     'species': os.path.join(BASE, 'data/biota_dataset/species.csv'),
# }

# multi_source = {
#     'event': os.path.join(BASE, 'data/protoDT/annual_budburst_per_tree.csv'),
#     'occurrence': os.path.join(BASE, 'data/protoDT/budburst_climwin_input.csv'),
#     'temp': os.path.join(BASE, 'data/protoDT/temp_climwin_input.csv'),
# }

# multi_source = {
#     'event': os.path.join(BASE, 'data/cropharvest/cropharvest_cleaned_global.csv')
# }

multi_source = {
    'event': os.path.join(BASE, 'data/S2BMS/bms_presence_y-2018-2019_th-200.csv'),
}

In [14]:
data_context = create_context(
    source=multi_source,
    name='biota_multi'
)
# metadata_standard=METADATA_STANDARDS['spatial_ecological']
metadata_standard = METADATA_STANDARDS[STANDARD_NAME]
orchestrator = Orchestrator(topology_name='single')
plan = orchestrator.generate_plan(
    context=data_context,
    metadata_standard=metadata_standard,
    metadata_standard_name=STANDARD_NAME,
)

2026-06-25 11:57:43,978 - INFO - root - PlanExecutor initialized with topology: single
2026-06-25 11:57:43,979 - INFO - root -   Players per step: 1
2026-06-25 11:57:43,980 - INFO - root -   Debate rounds: 0
2026-06-25 11:57:43,980 - INFO - root -   Player pool: ['dataset_schema_preview', 'data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
2026-06-25 11:57:43,980 - INFO - root - Orchestrator initialized with topology: single
2026-06-25 11:57:43,981 - INFO - root - [ui] planning
2026-06-25 11:57:43,981 - INFO - root - ============================================================
2026-06-25 11:57:43,981 - INFO - root - GENERATING PLAN
2026-06-25 11:57:43,981 - INFO - root - Context: biota_multi
2026-06-25 11:57:43,982 - INFO - root - Context type: single_csv
2026-06-25 11:57:43,982 - INFO - root - Classified planning type: single_csv
2026-06-25 11:57:43,982 - INFO - root - Resources: ['event']
2026-06-25 11:57:43,982 - I

In [15]:
# Validate plan dataflow
from src.orchestrator.utils import validate_plan_dataflow

if plan:
    # Convert to dict list for validation
    plan_dicts = plan.to_dict_list()
    
    is_valid, message = validate_plan_dataflow(plan_dicts)
    
    if is_valid:
        print(f"✅ {message}")
    else:
        print(f"❌ {message}")
else:
    print("No plan to validate.")

✅ Plan dataflow is valid.


In [16]:
plan_dicts

[{'task': 'get_columns_with_first_row',
  'player': 'dataset_schema_preview',
  'rationale': 'Capture the schema and representative data to inform recordset and field definitions.',
  'target_resources': [],
  'inputs': {},
  'outputs': ['schema_preview']},
 {'task': 'get_context_overview',
  'player': 'relationship_analyst',
  'rationale': 'Analyze the dataset structure to identify primary keys and relationships for the recordset definition.',
  'target_resources': [],
  'inputs': {'schema': 'schema_preview'},
  'outputs': ['relationship_data']},
 {'task': 'detect_temporal_columns and detect_spatial_columns',
  'player': 'spatial_temporal_specialist',
  'rationale': 'Identify spatial and temporal columns to extract coverage and resolution metadata.',
  'target_resources': [],
  'inputs': {'schema': 'schema_preview'},
  'outputs': ['spatial_temporal_data']},
 {'task': 'generate_croissant_metadata',
  'player': 'croissant_metadata_generator',
  'rationale': 'Generate the final Croissant

In [17]:
plan.pretty_print()

Plan Steps:
Step 0: get_columns_with_first_row
  Rationale: Capture the schema and representative data to inform recordset and field definitions.
  Required Artifacts: {}
  Produced Artifacts: ['schema_preview']
Step 1: get_context_overview
  Rationale: Analyze the dataset structure to identify primary keys and relationships for the recordset definition.
  Required Artifacts: {'schema': 'schema_preview'}
  Produced Artifacts: ['relationship_data']
Step 2: detect_temporal_columns and detect_spatial_columns
  Rationale: Identify spatial and temporal columns to extract coverage and resolution metadata.
  Required Artifacts: {'schema': 'schema_preview'}
  Produced Artifacts: ['spatial_temporal_data']
Step 3: generate_croissant_metadata
  Rationale: Generate the final Croissant metadata JSON using the gathered schema, relationship, and spatial-temporal artifacts.
  Required Artifacts: {'metadata_standard': 'metadata_standard', 'schema': 'schema_preview', 'relationships': 'relationship_data'

In [18]:
from src.orchestrator.plan_executor import PlanExecutor
from src.tools.context_tools import register_context

context_key = "ctx_biota_multi"
register_context(context_key, data_context)
executor = PlanExecutor(topology_name="single")
result = executor.execute(
    plan=plan,
    context=data_context,
    context_key=context_key,
    metadata_standard=metadata_standard,
    metadata_standard_name=STANDARD_NAME
)

2026-06-25 11:57:48,417 - INFO - root - PlanExecutor initialized with topology: single
2026-06-25 11:57:48,417 - INFO - root -   Players per step: 1
2026-06-25 11:57:48,417 - INFO - root -   Debate rounds: 0
2026-06-25 11:57:48,418 - INFO - root -   Player pool: ['dataset_schema_preview', 'data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
2026-06-25 11:57:48,418 - INFO - root - Using structured output schema: CroissantStandardSubsetMetadata
2026-06-25 11:57:48,418 - INFO - root - ============================================================
2026-06-25 11:57:48,419 - INFO - root - STARTING PLAN EXECUTION
2026-06-25 11:57:48,419 - INFO - root - Context: biota_multi
2026-06-25 11:57:48,419 - INFO - root - Type: single_csv
2026-06-25 11:57:48,419 - INFO - root - Resources: ['event']
2026-06-25 11:57:48,419 - INFO - root - Steps: 4
2026-06-25 11:57:48,419 - INFO - root - ====================================================

In [19]:
from pprint import pprint
pprint(result.final_workspace['metadata_output'])

{'description': 'A dataset containing butterfly species observation counts '
                'across various locations in the UK, including visit frequency '
                'and geographic coordinates.',
 'filesets': {'excludes': [], 'includes': ['event.csv']},
 'inLanguage': ['en'],
 'keywords': ['butterfly', 'biodiversity', 'UKBMS', 'observation', 'ecology'],
 'name': 'UKBMS Butterfly Observation Dataset',
 'recordsets': [{'examples': ['{"Unnamed: 0": 0, "tuple_coords": "(-7.824283, '
                              '54.259247)", "n_visits": 1, "name_loc": '
                              '"UKBMS_loc-1044", "Melanargia galathea": 0.0, '
                              '"Pieris napi": 0.0, "Aphantopus hyperantus": '
                              '0.0}'],
                 'fields': [],
                 'key': 'name_loc',
                 'source': 'event.csv'}],
 'spatial': None,
 'spatialCoverage': {'max_lat': 58.606506,
                     'max_lon': 1.693422,
                     'min_

In [20]:
executor.list_tools_called()

['get_columns_with_first_row',
 'get_context_overview',
 'detect_temporal_columns',
 'detect_spatial_columns',
 'get_spatial_extent_from_tuple_column',
 'get_field_names',
 'get_field_types']

In [23]:
executor.find_tool_calls('detect_spatial_columns')

[{'agent': 'spatial_temporal_specialist_1',
  'input': {'context_key': 'ctx_biota_multi', 'resource': 'event'},
  'output': {'resource': 'event',
   'spatial_column_count': 64,
   'spatial_columns': {'tuple_coords': {'name_suggests_spatial': True,
     'detected_type': 'two_float_tuple_string',
     'sample_values': ['(-7.824283, 54.259247)',
      '(-7.457325, 54.15933)',
      '(-6.949025, 54.883646)',
      '(-6.753205, 55.169256)',
      '(-6.405594, 55.228089)'],
     'tuple_order_hint': 'lon_lat'},
    'Melanargia galathea': {'name_suggests_spatial': True,
     'detected_type': 'possible_latitude',
     'sample_values': ['0.0', '0.0', '0.0', '0.0', '0.0']},
    'Pieris napi': {'name_suggests_spatial': False,
     'detected_type': 'possible_latitude',
     'sample_values': ['0.696969696969697',
      '0.4193548387096774',
      '0.3636363636363636',
      '0.4482758620689655',
      '0.71875']},
    'Aphantopus hyperantus': {'name_suggests_spatial': False,
     'detected_type': 'p

In [22]:
pprint(result.final_workspace['spatial_temporal_analysis'])

KeyError: 'spatial_temporal_analysis'

In [ ]:
print(list(result.final_workspace.keys()))
